<a href="https://colab.research.google.com/github/M-AyyanAsif/DeepLearning_Concepts/blob/main/Hyperparameter_Tunning_in_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("/data.csv")
df.head(5)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [ ]:
df.drop(columns=['id','Gender','Vehicle_Age','Vehicle_Damage'],inplace=True)
df.head(5)

,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,44,1,28.0,0,40454.0,26.0,217,1
1,76,1,3.0,0,33536.0,26.0,183,0
2,47,1,28.0,0,38294.0,26.0,27,1
3,21,1,11.0,1,28619.0,152.0,203,0
4,29,1,41.0,1,27496.0,152.0,39,0


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Select only numerical columns
df.select_dtypes(include='number').corr()['Response']


,Response
Age,0.111147
Driving_License,0.010155
Region_Code,0.010570
Previously_Insured,-0.341170
Annual_Premium,0.022575
Policy_Sales_Channel,-0.139042
Vintage,-0.001050
Response,1.000000


In [ ]:
X=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras import layers
from keras.layers import Dense ,Dropout

In [ ]:
!pip install -q keras-tuner

In [ ]:
import keras_tuner as kt

# Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Searching For Hyper Model

In [ ]:
def build_model(hp):
  model=keras.Sequential()

  # Input shape should be a tuple, even for a single dimension
  model.add(layers.Input(shape=(X_train.shape[1],)))

  for i in range(hp.Int('num_layers',1,5)):
    model.add(layers.Dense(
        units=hp.Int(
            'units_'+str(i),
            min_value=32,
            max_value=512, # Corrected typo from max_vlaue
            step=32),
        activation=hp.Choice('activation_'+str(i),['relu','elu','tanh']) # Corrected syntax and made name unique
        )
    )
    model.add(
        layers.Dropout(
            rate=hp.Float('rate_'+str(i),min_value=0.0,max_value=0.5,step=0.1)
        )

    )
  # The output layer should be added after all hidden layers
  model.add(Dense(1,activation='sigmoid'))

  # Optimizer and model compilation should happen once after defining the full model architecture
  optimizer_name = hp.Choice('optimizer', values=['adam', 'rmsprop'])
  learning_rate = hp.Float('lr', 1e-4, 1e-2, sampling='log')

  if optimizer_name == 'adam':
      optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
  else: # rmsprop
      optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

  # The function should return the built model, not the function itself
  return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='ann_tuning',
    project_name='keras_ann'
)


In [ ]:
from keras.callbacks import EarlyStopping

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


In [ ]:
tuner.search(
    X_train,
    y_train,
    validation_data=[X_test,y_test],
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Trial 20 Complete [00h 03m 47s]
val_accuracy: 0.8796935081481934

Best val_accuracy So Far: 0.8797197937965393
Total elapsed time: 03h 11m 24s


In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:")
for param in best_hp.values:
    print(param, ":", best_hp.get(param))


Best Hyperparameters:
num_layers : 4
units_0 : 512
activation_0 : elu
rate_0 : 0.2
optimizer : rmsprop
lr : 0.001022669883203695
units_1 : 352
activation_1 : tanh
rate_1 : 0.1
units_2 : 64
activation_2 : elu
rate_2 : 0.4
units_3 : 448
activation_3 : tanh
rate_3 : 0.0
units_4 : 512
activation_4 : relu
rate_4 : 0.4


In [ ]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    X_train,
    y_train,
    validation_data=[X_test,y_test],
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")
